# NOT training with ICNN 

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import Transport, Critic, RF_Transport, RF_Critic, ICNNCritic
from src.utils import cost, show_mapping , grad_norm, cosine_lr, plot_3d_function, ema_np, plot_loss, plot_grad_norm
from src.train import train_gdmax, train_extragradient


In [ ]:
# source distribution μ
def sample_mu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.zeros(batch_size, 1, device=device)         # y = 0
    return torch.cat([x, y], dim=1)


# target distribution ν
def sample_nu(batch_size, device="cpu"):
    x = torch.zeros(batch_size, 1, device=device)         # y = 0
    y = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    return torch.cat([x, y], dim=1)

In [ ]:
T = Transport().to(device)
f = ICNNCritic().to(device)

loss_hist = []
grad_T_hist = []
grad_f_hist = []

## GDmax Training

In [ ]:
history = train_gdmax(
    T, f, sample_mu, sample_nu, cost,
    n_steps=20000, lr_T=1e-5, lr_f=1e-5, K=10,
    batch_size_x=1024, batch_size_y=1024,
    lr_schedule=lambda step: (cosine_lr(step, 2000, 1e-4, 0),) * 2,
    callback=lambda step, T, f: show_mapping(T, sample_mu, sample_nu, f=f, contour=True),
)
loss_hist += history["loss"]
grad_T_hist += history["grad_T"]
grad_f_hist += history["grad_f"]

## Extragradient Training

In [ ]:
history = train_extragradient(
    T, f, sample_mu, sample_nu, cost,
    n_steps=100000, lr_T=1e-4, lr_f=1e-4,
    batch_size_x=1024, batch_size_y=1024,
    callback=lambda step, T, f: show_mapping(T, sample_mu, sample_nu, f=f, contour=True),
)
loss_hist += history["loss"]
grad_T_hist += history["grad_T"]
grad_f_hist += history["grad_f"]

## Plot Results

In [ ]:
plot_loss(loss_hist)

In [ ]:
show_mapping(T,sample_mu,sample_nu, option = False, f=f, contour=True, legend = False)

In [ ]:
plot_3d_function(f)